# Visible Data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import glob
from astropy.io import fits
from scipy import stats
from scipy.stats import norm

bias_data = []
bias_headers = []

for filename in glob.glob("../data/m13/Bias*"):
    data = fits.getdata(filename)
    header = fits.getheader(filename)

    bias_data.append(data)
    bias_headers.append(header)

median_bias = np.median(bias_data, axis=0)

# Make dark combines

In [ ]:
dark_data_60s = []
dark_data_12s = []
dark_data_3s = []
dark_data_1s = []

for filename in glob.glob("../data/m13/Dark-60*"):
    data = fits.getdata(filename).astype(np.float64)
    data -= median_bias
    dark_data_60s.append(data)

master_dark_60s = np.median(dark_data_60s, axis=0)

for filename in glob.glob("../data/m13/Dark-12*"):
    data = fits.getdata(filename).astype(np.float64)
    data -= median_bias
    dark_data_12s.append(data)

master_dark_12s = np.median(dark_data_12s, axis=0)

for filename in glob.glob("../data/m13/Dark-3*"):
    data = fits.getdata(filename).astype(np.float64)
    data -= median_bias
    dark_data_3s.append(data)

master_dark_3s = np.median(dark_data_3s, axis=0)

for filename in glob.glob("../data/m13/Dark-1s*"):
    data = fits.getdata(filename).astype(np.float64)
    data -= median_bias
    dark_data_1s.append(data)

master_dark_1s = np.median(dark_data_1s, axis=0)

#cutoff = 1050

#hot_mask = master_dark <= cutoff


In [ ]:
flat_v_data = []
for filename in glob.glob("../data/m13/Flat-V*"):
    data = fits.getdata(filename).astype(np.float64)
    data -= median_bias 
    data -= master_dark_3s 
    flat_v_data.append(data)

master_v_flat = np.median(flat_v_data, axis=0)
master_v_flat /= stats.mode(master_v_flat.flatten())[0]

/var/folders/74/9yc0dt9d53z15d2qpmgpxv4h0000gn/T/ipykernel_2222/3241251790.py:9: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  master_v_flat /= stats.mode(master_v_flat.flatten())[0]


In [ ]:
science_data = []
for filename in glob.glob("../data/m13/M13-V-60*"):
    data = fits.getdata(filename)
    science_data.append(data)

reduced_data = []
for i in range(len(science_data)):
    raw_data = science_data[i].astype(np.float64)
    raw_data -= median_bias
    raw_data -= master_dark_60s
    raw_data = raw_data / master_v_flat

    reduced_data.append(raw_data)

In [ ]:
import astroalign as aa

reference = reduced_data[0]

aligned_data = [reference]

for i, image in enumerate(reduced_data[1:], start=1):
    print(f"Aligning image {i + 1}/{len(reduced_data)}...")

    aligned_image, footprint = aa.register(
        image,
        reference,
        fill_value=np.nan
    )

    aligned_data.append(aligned_image)

# average the aligned images
stacked_image = np.nanmean(aligned_data, axis=0)

for filename in glob.glob("../data/m13/M13-V-60*"):
    header = fits.getheader(filename)

header["NCOMBINE"] = (
    len(aligned_data),
    "Number of images combined"
)

header["STACK"] = (
    "MEDIAN",
    "Stacking method"
)

fits.writeto(
    "../data/m13/M13-V-stacked.fits",
    stacked_image,
    header=header,
    overwrite=True
)

print("Saved stacked image to ../data/m13/M13-V-stacked.fits")

Aligning image 2/2...


<frozen importlib._bootstrap>:228: FutureWarning: 
The `sep-pjw` package has reverted to the original `sep` package name. Bug
fixes and additional enhancements will not be released for `sep-pjw`, and
users of this package should update their dependencies to `sep>=1.4.0`.



Saved stacked image to m13/M13-V-stacked.fits


# Blue Data

In [ ]:
flat_b_data = []
for filename in glob.glob("../data/m13/Flat-B*"):
    data = fits.getdata(filename).astype(np.float64)
    data -= median_bias
    data -= master_dark_12s 
    flat_b_data.append(data)

master_b_flat = np.median(flat_b_data, axis=0)
master_b_flat /= stats.mode(master_b_flat.flatten())[0]

/var/folders/74/9yc0dt9d53z15d2qpmgpxv4h0000gn/T/ipykernel_2222/369649309.py:9: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  master_b_flat /= stats.mode(master_b_flat.flatten())[0]


In [ ]:
science_data = []
for filename in glob.glob("../data/m13/M13-B-60*"):
    data = fits.getdata(filename)
    science_data.append(data)

reduced_data = []
for i in range(len(science_data)):
    raw_data = science_data[i].astype(np.float64)
    raw_data -= median_bias
    raw_data -= master_dark_60s
    raw_data = raw_data / master_v_flat

    reduced_data.append(raw_data)

In [ ]:
reference = reduced_data[0]

aligned_data = [reference]

for i, image in enumerate(reduced_data[1:], start=1):

    print(f"Aligning image {i + 1}/{len(reduced_data)}...")

    aligned_image, footprint = aa.register(
        image,
        reference,
        fill_value=np.nan
    )

    aligned_data.append(aligned_image)

# average the aligned images
stacked_image = np.nanmean(aligned_data, axis=0)

for filename in glob.glob("../data/m13/M13-B-60*"):
    header = fits.getheader(filename)

header["NCOMBINE"] = (
    len(aligned_data),
    "Number of images combined"
)

header["STACK"] = (
    "MEDIAN",
    "Stacking method"
)

fits.writeto(
    "../data/m13/M13-B-stacked.fits",
    stacked_image,
    header=header,
    overwrite=True
)

print("Saved stacked image to ../data/m13/M13-B-stacked.fits")

Aligning image 2/2...
Saved stacked image to m13/M13-B-stacked.fits


# Red Data

In [ ]:
flat_r_data = []
for filename in glob.glob("../data/m13/Flat-R*"):
    data = fits.getdata(filename).astype(np.float64)
    data -= median_bias  
    data -= master_dark_1s  
    flat_r_data.append(data)

master_r_flat = np.median(flat_r_data, axis=0)
master_r_flat /= stats.mode(master_r_flat.flatten())[0]

/var/folders/74/9yc0dt9d53z15d2qpmgpxv4h0000gn/T/ipykernel_2222/949171537.py:9: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  master_r_flat /= stats.mode(master_r_flat.flatten())[0]


In [ ]:
science_data = []
for filename in glob.glob("../data/m13/M13-R-60*"):
    data = fits.getdata(filename)
    science_data.append(data)

reduced_data = []
for i in range(len(science_data)):
    raw_data = science_data[i].astype(np.float64)
    raw_data -= median_bias
    raw_data -= master_dark_60s
    raw_data = raw_data / master_r_flat

    reduced_data.append(raw_data)

In [ ]:
reference = reduced_data[0]

aligned_data = [reference]

for i, image in enumerate(reduced_data[1:], start=1):

    print(f"Aligning image {i + 1}/{len(reduced_data)}...")

    aligned_image, footprint = aa.register(
        image,
        reference,
        fill_value=np.nan
    )

    aligned_data.append(aligned_image)

# average the aligned images
stacked_image = np.nanmean(aligned_data, axis=0)

for filename in glob.glob("../data/m13/M13-R-60*"):
    header = fits.getheader(filename)

header["NCOMBINE"] = (
    len(aligned_data),
    "Number of images combined"
)

header["STACK"] = (
    "MEDIAN",
    "Stacking method"
)

fits.writeto(
    "../data/m13/M13-R-stacked.fits",
    stacked_image,
    header=header,
    overwrite=True
)

print("Saved stacked image to ../data/m13/M13-R-stacked.fits")

Aligning image 2/2...
Saved stacked image to m13/M13-R-stacked.fits
